# QLoRA demo — MLX (Apple Silicon)

MLX port of `QLoRA_demo.ipynb` (CUDA + bitsandbytes + PEFT). Same model, data, LoRA hyper-parameters and training schedule.

| CUDA notebook | MLX notebook |
|---|---|
| `BitsAndBytesConfig` (4-bit NF4) | `mlx_lm.utils.quantize_model` (4-bit affine, group size 64) |
| `prepare_model_for_kbit_training` + `get_peft_model` | `model.freeze()` + `linear_to_lora_layers` |
| `Trainer` / `paged_adamw_8bit` | plain MLX loop / `optim.AdamW` |
| `model.generate` | `mlx_lm.generate` |

Note: `mlx-lm` loads only the language model of Qwen3.5 and drops the vision tower. The CUDA version kept the vision tower unquantized and never used it, so the text-only result is the same.

**Setup** (once, from this folder):

```bash
uv venv .venv && uv pip install --python .venv/bin/python mlx-lm ipykernel
```

Then select the `.venv` kernel for this notebook.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten
from mlx_lm import load, generate
from mlx_lm.utils import quantize_model
from mlx_lm.tuner.utils import linear_to_lora_layers, print_trainable_parameters

MODEL_ID = "Qwen/Qwen3.5-2B"

model, tok, config = load(MODEL_ID, return_config=True)

# 4-bit quantize the frozen base weights (the "Q" in QLoRA).
# Like llm_int8_skip_modules=["lm_head"], keep the (possibly tied) embedding / lm_head in full precision.
skip = lambda path, _: not (path.endswith("lm_head") or path.endswith("embed_tokens"))
model, config = quantize_model(model, config, group_size=64, bits=4, quant_predicate=skip)

In [ ]:
SYSTEM = "Extract the student's registration number from the ID card. Respond with JSON only."

data = {
    "input": [
        """# ANNA UNIVERSITY
## Student Identity Card
**Name:** Priya Raman
**Department:** B.E. Computer Science
**Register No.:** 312221104087
**Batch:** 2021–2025
**Blood Group:** O+
**Emergency Contact:** 9840012345""",

        """| Field | Value |
|---|---|
| Student Name | Arjun Mehta |
| Programme | B.Tech Mechanical |
| Reg. ID | RA2111003010245 |
| Library Card | LIB-55821 |
| Valid Till | 05/2025 |""",

        """### SRM Institute of Science and Technology
Name: Kavya S
Course: MBA
Enrollment Number: 21MBA0193
Admission No: ADM/2021/7741
Hostel: Block C, Room 214""",
    ],
    "output": [
        '{"registration_number": "312221104087"}',
        '{"registration_number": "RA2111003010245"}',
        '{"registration_number": "21MBA0193"}',
    ],
}

TEST_CARD = """## VIT University — Student ID
**Name:** Rohan K
**Branch:** ECE
**Reg No:** 22BEC1043
**Mobile:** 9876501234
**Bus Route:** 14B"""

In [ ]:
def chat_prompt(text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def extract(text, max_new_tokens=64):
    # greedy decoding by default (same as do_sample=False)
    return generate(model, tok, prompt=chat_prompt(text), max_tokens=max_new_tokens).strip()

print("BEFORE:", extract(TEST_CARD))

In [ ]:
MAX_LEN = 512

def build(inp, out):
    p_ids = tok.encode(chat_prompt(inp), add_special_tokens=False)
    a_ids = tok.encode(out + "<|im_end|>", add_special_tokens=False)
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]   # loss only on the answer
    return {"input_ids": ids, "labels": labels}

ds = [build(i, o) for i, o in zip(data["input"], data["output"])]

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, v: seq + [v] * (m - len(seq))
    return (
        mx.array([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
        mx.array([pad(b["labels"], -100) for b in batch]),
    )

In [ ]:
LORA = {"rank": 16, "scale": 32 / 16, "dropout": 0.05}   # scale = lora_alpha / r

model.freeze()                                            # base weights stay frozen (and 4-bit)
linear_to_lora_layers(model, len(model.layers), LORA)     # every linear layer in every block: attention + DeltaNet + MLP
print_trainable_parameters(model)

EPOCHS = 10            # deliberately overfit 3 samples to prove the loop works
BATCH_SIZE = 1
optimizer = optim.AdamW(learning_rate=2e-4, weight_decay=0.0)   # constant LR

def loss_fn(model, input_ids, labels):
    # shift: predict token t+1 from tokens <= t
    logits = model(input_ids[:, :-1])
    targets = labels[:, 1:]
    mask = targets != -100
    ce = nn.losses.cross_entropy(logits, mx.where(mask, targets, 0)) * mask
    return ce.astype(mx.float32).sum() / mask.sum()

loss_and_grad = nn.value_and_grad(model, loss_fn)

model.train()
step = 0
for epoch in range(EPOCHS):
    for i in range(0, len(ds), BATCH_SIZE):
        input_ids, labels = collate(ds[i : i + BATCH_SIZE])
        loss, grads = loss_and_grad(model, input_ids, labels)
        optimizer.update(model, grads)
        mx.eval(model.parameters(), optimizer.state, loss)
        step += 1
        print(f"step {step:3d} | epoch {epoch + 1:2d} | loss {loss.item():.4f}")

In [ ]:
import json
from pathlib import Path

model.eval()
print("AFTER:", extract(TEST_CARD))

# adapter only, ~tens of MB; reload with mlx_lm.load(MODEL_ID, adapter_path=...)
out = Path("qwen35-2b-idcard-lora")
out.mkdir(exist_ok=True)
mx.save_safetensors(str(out / "adapters.safetensors"), dict(tree_flatten(model.trainable_parameters())))
(out / "adapter_config.json").write_text(json.dumps({
    "fine_tune_type": "lora",
    "num_layers": len(model.layers),
    "lora_parameters": LORA,
}, indent=2))
sorted(p.name for p in out.iterdir())